## Hybrid Recommender System

The goal of this experiment is to investigate whether combining two collaborative filtering approaches can improve recommendation performance on the MovieLens 100K dataset.

The hybrid recommender combines:

- **ItemKNN**, a neighborhood-based collaborative filtering method that recommends items based on item-item similarity.
- **SLIMElastic**, a sparse linear recommendation model that learns relationships between items using Elastic Net regularization.

The motivation for combining these models is that they capture item relationships in different ways. ItemKNN relies directly on similarity between items, while SLIMElastic learns sparse item-item coefficients from user interaction data. Combining their predictions may therefore provide complementary information and improve recommendation quality.

## Hybrid Method

The hybrid model combines the prediction scores produced by ItemKNN and SLIMElastic using a weighted linear combination:

\[
S_{\text{hybrid}} = (1-\alpha)S_{\text{ItemKNN}} + \alpha S_{\text{SLIM}}
\]

where:

- \(S_{\text{ItemKNN}}\) is the score produced by ItemKNN.
- \(S_{\text{SLIM}}\) is the score produced by SLIMElastic.
- \(\alpha\) controls the contribution of each model.

When \(\alpha = 0\), the recommender is equivalent to ItemKNN, while \(\alpha = 1\) corresponds to SLIMElastic only.

Before combining the scores, min-max normalization is applied separately to the score vectors of both models. This prevents differences in the models' score scales from disproportionately influencing the weighted combination.

Items that the user has already interacted with are masked before generating the final Top-10 recommendations.

## Evaluation Methodology

The MovieLens 100K dataset was divided into training, validation, and test sets using an 80/10/10 split.

The models were evaluated using full-sort evaluation, where candidate items are ranked for each user and previously interacted items are excluded from the recommendation list.

The following Top-10 ranking metrics were used:

- **Recall@10**: measures the proportion of relevant items successfully retrieved.
- **Precision@10**: measures the proportion of recommended items that are relevant.
- **MRR@10**: measures how highly the first relevant item appears in the ranking.
- **NDCG@10**: evaluates ranking quality while giving greater importance to relevant items appearing near the top.
- **Hit@10**: measures whether at least one relevant item appears in the Top-10 recommendations.

NDCG@10 was used as the primary metric for selecting the hybrid weighting parameter.

Importantly, the value of \(\alpha\) was selected using the validation set rather than the test set. The test set was only used for the final evaluation after the optimal value of \(\alpha\) had been selected.

In [ ]:
pip install --force-reinstall "numpy==1.23.5"

In [ ]:
import sys
import numpy as np
import torch
import recbole

print("Python :", sys.executable)
print("NumPy  :", np.__version__)
print("PyTorch:", torch.__version__)
print("RecBole:", recbole.__version__)

from recbole.quick_start import load_data_and_model
from recbole.utils.case_study import full_sort_topk

In [ ]:
from recbole.quick_start import run_recbole

itemknn_result = run_recbole(
    model="ItemKNN",
    dataset="ml-100k",
)

slim_result = run_recbole(
    model="SLIMElastic",
    dataset="ml-100k",
)

In [ ]:
from recbole.quick_start import load_data_and_model

ITEMKNN_CHECKPOINT = "saved/ItemKNN-Oct-07-2026_17-09-44.pth"
SLIM_CHECKPOINT = "saved/SLIMElastic-Oct-07-2026_17-09-54.pth"

(
    itemknn_config,
    itemknn_model,
    itemknn_dataset,
    itemknn_train,
    itemknn_valid,
    itemknn_test,
) = load_data_and_model(ITEMKNN_CHECKPOINT)

(
    slim_config,
    slim_model,
    slim_dataset,
    slim_train,
    slim_valid,
    slim_test,
) = load_data_and_model(SLIM_CHECKPOINT)

itemknn_model.eval()
slim_model.eval()

In [ ]:
import numpy as np
import torch
from recbole.data.interaction import Interaction

In [ ]:
def normalize_rows(scores):
    scores = scores.clone()

    min_vals = scores.min(dim=1, keepdim=True).values
    max_vals = scores.max(dim=1, keepdim=True).values

    return (
        (scores - min_vals)
        / (max_vals - min_vals + 1e-12)
    )

In [ ]:
def metrics_at_k(recommended, relevant, k=10):
    recommended = recommended[:k]
    relevant = set(relevant)

    hits = [
        1 if item in relevant else 0
        for item in recommended
    ]

    # Precision@K
    precision = sum(hits) / k

    # Recall@K
    recall = (
        sum(hits) / len(relevant)
        if relevant else 0.0
    )

    # Hit@K
    hit = 1.0 if sum(hits) > 0 else 0.0

    # MRR@K
    mrr = 0.0

    for rank, is_hit in enumerate(hits, start=1):
        if is_hit:
            mrr = 1.0 / rank
            break

    # NDCG@K
    dcg = sum(
        is_hit / np.log2(rank + 2)
        for rank, is_hit in enumerate(hits)
    )

    ideal_hits = min(len(relevant), k)

    idcg = sum(
        1.0 / np.log2(rank + 2)
        for rank in range(ideal_hits)
    )

    ndcg = dcg / idcg if idcg > 0 else 0.0

    return {
        "recall": recall,
        "mrr": mrr,
        "ndcg": ndcg,
        "hit": hit,
        "precision": precision,
    }

## Hybrid Method

The hybrid model combines the prediction scores produced by ItemKNN and SLIMElastic using a weighted linear combination:

\[
S_{\text{hybrid}} = (1-\alpha)S_{\text{ItemKNN}} + \alpha S_{\text{SLIM}}
\]

where:

- \(S_{\text{ItemKNN}}\) is the score produced by ItemKNN.
- \(S_{\text{SLIM}}\) is the score produced by SLIMElastic.
- \(\alpha\) controls the contribution of each model.

When \(\alpha = 0\), the recommender is equivalent to ItemKNN, while \(\alpha = 1\) corresponds to SLIMElastic only.

Before combining the scores, min-max normalization is applied separately to the score vectors of both models. This prevents differences in the models' score scales from disproportionately influencing the weighted combination.

Items that the user has already interacted with are masked before generating the final Top-10 recommendations.

In [ ]:
def evaluate_hybrid(eval_data, alpha=0.5, k=10):
    results = []

    itemknn_model.eval()
    slim_model.eval()

    with torch.no_grad():

        for batch in eval_data:

            (
                interaction,
                history_index,
                positive_u,
                positive_i,
            ) = batch

            interaction = interaction.to(
                itemknn_config["device"]
            )

            # --------------------------------
            # Predictions
            # --------------------------------

            item_scores = (
                itemknn_model
                .full_sort_predict(interaction)
            )

            slim_scores = (
                slim_model
                .full_sort_predict(interaction)
            )

            item_scores = item_scores.view(
                len(interaction),
                -1
            )

            slim_scores = slim_scores.view(
                len(interaction),
                -1
            )

            # --------------------------------
            # Normalize
            # --------------------------------

            item_scores = normalize_rows(
                item_scores
            )

            slim_scores = normalize_rows(
                slim_scores
            )

            # --------------------------------
            # Hybrid
            # --------------------------------

            scores = (
                (1 - alpha) * item_scores
                + alpha * slim_scores
            )

            # Padding item
            scores[:, 0] = -torch.inf

            # --------------------------------
            # Mask user's history
            # --------------------------------

            if history_index is not None:

                history_u, history_i = (
                    history_index
                )

                scores[
                    history_u,
                    history_i
                ] = -torch.inf

            # --------------------------------
            # Top-K
            # --------------------------------

            top_items = torch.topk(
                scores,
                k=k,
                dim=1
            ).indices.cpu()

            # --------------------------------
            # Ground truth
            # --------------------------------

            ground_truth = {}

            for u_idx, item_id in zip(
                positive_u.cpu().tolist(),
                positive_i.cpu().tolist()
            ):

                ground_truth.setdefault(
                    int(u_idx),
                    set()
                ).add(int(item_id))

            # --------------------------------
            # Metrics
            # --------------------------------

            for u_idx, relevant in ground_truth.items():

                recommended = (
                    top_items[u_idx]
                    .tolist()
                )

                result = metrics_at_k(
                    recommended,
                    relevant,
                    k=k
                )

                results.append(result)

    # --------------------------------
    # Average across users
    # --------------------------------

    final = {
        metric: np.mean([
            result[metric]
            for result in results
        ])
        for metric in [
            "recall",
            "mrr",
            "ndcg",
            "hit",
            "precision",
        ]
    }

    return final

In [ ]:
itemknn_check = evaluate_hybrid(
    itemknn_test,
    alpha=0.0,
    k=10
)

print("ItemKNN sanity check")

for metric, value in itemknn_check.items():
    print(f"{metric}@10: {value:.4f}")

In [ ]:
slim_test = evaluate_hybrid(
    itemknn_test,
    alpha=1.0,
    k=10
)

print("SLIMElastic Test Results")

for metric, value in slim_test.items():
    print(f"{metric}@10: {value:.4f}")

In [ ]:
alphas = np.arange(0.0, 1.01, 0.1)

validation_results = []

for alpha in alphas:

    result = evaluate_hybrid(
        itemknn_valid,
        alpha=float(alpha),
        k=10
    )

    validation_results.append({
        "alpha": float(alpha),
        **result
    })

    print(
        f"alpha={alpha:.1f} | "
        f"Recall={result['recall']:.4f} | "
        f"MRR={result['mrr']:.4f} | "
        f"NDCG={result['ndcg']:.4f} | "
        f"Hit={result['hit']:.4f} | "
        f"Precision={result['precision']:.4f}"
    )

In [ ]:
import pandas as pd

validation_df = pd.DataFrame(validation_results)

validation_df

In [ ]:
best_idx = validation_df["ndcg"].idxmax()
best_validation = validation_df.loc[best_idx]

best_alpha = float(best_validation["alpha"])

print("Best alpha from VALIDATION:")
print(f"Alpha: {best_alpha:.1f}")
print(f"NDCG@10: {best_validation['ndcg']:.4f}")

In [ ]:
final_test = evaluate_hybrid(
    itemknn_test,
    alpha=best_alpha,
    k=10
)

print(f"Final Hybrid Test Results (alpha={best_alpha:.1f})")

for metric, value in final_test.items():
    print(f"{metric}@10: {value:.4f}")